# 34 - TCGA-BRCA Patient-Level Unlearning

Patient-level unlearning on TCGA-BRCA. Unlike the structured forget set (all 158 Basal),
these forget sets contain small groups of specific patients (5, 10, 20 Basal patients).

**Key difference from NB33:** Matched negatives are recomputed per forget set via k-NN
in the baseline model's latent space (per mia-evaluation skill protocol).

**Methods:** Fisher scrubbing, Retain fine-tune, Gradient ascent.

**Dependencies:** `data/tcga_brca/tcga_brca_processed.h5ad`, `outputs/tcga_brca/baseline/best_model.pt`, `outputs/tcga_brca/split_patient_{5,10,20}.json`

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import json
import scanpy as sc
from pathlib import Path
from sklearn.neighbors import NearestNeighbors

from utils import set_global_seed, GLOBAL_SEED, DEVICE, Timer
from vae import VAE
from attacker import MLPAttacker, extract_vae_features, build_attack_features
from attacker_eval import matched_negative_evaluation
from train_fisher_unlearn import AnnDataDataset, compute_fisher_diagonal, fisher_scrub, retain_finetune
from train_retain_finetune import train_retain_finetune
from train_gradient_ascent import train_gradient_ascent
from torch.utils.data import DataLoader

set_global_seed(GLOBAL_SEED)

DATA_DIR = Path('../data/tcga_brca')
BASELINE_DIR = Path('../outputs/tcga_brca/baseline')
OUTPUT_DIR = Path('../outputs/tcga_brca/patient_unlearning')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Seed: {GLOBAL_SEED}')
print(f'Device: {DEVICE}')

Seed: 42
Device: cpu


## Load Data and Baseline Model

In [2]:
adata = sc.read_h5ad(DATA_DIR / 'tcga_brca_processed.h5ad')
print(f'Data: {adata.shape}')

baseline_ckpt = torch.load(BASELINE_DIR / 'best_model.pt', map_location=DEVICE)
config = baseline_ckpt['config']
model_config = {k: v for k, v in config.items()
                if k in ['input_dim', 'latent_dim', 'hidden_dims', 'likelihood', 'dropout', 'use_layer_norm']}

def load_baseline():
    model = VAE(**model_config).to(DEVICE)
    model.load_state_dict(baseline_ckpt['model_state_dict'])
    return model

# Encode all samples with baseline model for k-NN matching
baseline_model = load_baseline()
baseline_model.eval()
X_all = adata.X
if hasattr(X_all, 'toarray'):
    X_all = X_all.toarray()
x_all = torch.FloatTensor(X_all).to(DEVICE)
with torch.no_grad():
    mu_all, _ = baseline_model.encode(x_all)
    z_all = mu_all.cpu().numpy()
print(f'Baseline latent: {z_all.shape}')

DATA_PATH = str(DATA_DIR / 'tcga_brca_processed.h5ad')
BASELINE_CKPT_PATH = str(BASELINE_DIR / 'best_model.pt')

Data: (1089, 978)
Baseline latent: (1089, 32)


## Evaluation Helpers

Per mia-evaluation skill:
- Matched negatives via k-NN (k=10) in baseline latent space, recomputed per forget set
- Fresh attacker per evaluation (spectral norm, [256,256], dropout 0.3)
- 69-dim v1 features
- Advantage = 2*|AUC - 0.5|

In [3]:
def compute_matched_negatives(forget_idx, unseen_idx, z_all, k=10):
    """Compute k-NN matched negatives for a specific forget set."""
    forget_z = z_all[forget_idx]
    unseen_z = z_all[unseen_idx]
    actual_k = min(k, len(unseen_z))
    nbrs = NearestNeighbors(n_neighbors=actual_k, algorithm='ball_tree').fit(unseen_z)
    distances, knn_idx = nbrs.kneighbors(forget_z)
    matched_local = np.unique(knn_idx.flatten())
    matched = [int(unseen_idx[i]) for i in matched_local]
    return matched

def extract_features_for_indices(vae_model, adata, indices, device):
    """Extract 69-dim v1 MIA features."""
    X = adata.X[indices]
    if hasattr(X, 'toarray'):
        X = X.toarray()
    x = torch.FloatTensor(X).to(device)
    lib = x.sum(dim=1, keepdim=True)
    vae_feats = extract_vae_features(vae_model, x, lib, device, requires_grad=False)
    return build_attack_features(vae_feats, variant='v1')

def evaluate_method(vae_model, forget_idx, matched_idx, method_name):
    """Full MIA evaluation per mia-evaluation skill protocol."""
    vae_model.eval()
    forget_feats = extract_features_for_indices(vae_model, adata, forget_idx, DEVICE)
    matched_feats = extract_features_for_indices(vae_model, adata, matched_idx, DEVICE)
    
    input_dim = forget_feats.shape[1]
    attacker = MLPAttacker(input_dim, [256, 256], dropout=0.3, use_spectral_norm=True).to(DEVICE)
    att_opt = optim.Adam(attacker.parameters(), lr=1e-3, weight_decay=1e-4)
    
    X_train = torch.cat([forget_feats, matched_feats]).to(DEVICE)
    y_train = torch.cat([torch.ones(len(forget_feats)),
                         torch.zeros(len(matched_feats))]).unsqueeze(1).to(DEVICE)
    
    for _ in range(100):
        attacker.train()
        att_opt.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(attacker(X_train), y_train)
        loss.backward()
        att_opt.step()
    
    attacker.eval()
    metrics = matched_negative_evaluation(attacker, forget_feats, matched_feats, device=DEVICE)
    auc = metrics['auc']
    advantage = 2 * abs(auc - 0.5)
    print(f'    {method_name}: AUC={auc:.3f}, advantage={advantage:.3f}')
    return {'auc': float(auc), 'advantage': float(advantage)}

print('Helpers ready')

Helpers ready


## Run Patient-Level Unlearning

In [4]:
all_patient_results = {}

for n_patients in [5, 10, 20]:
    print(f'\n{"="*60}')
    print(f'Patient-level forget set: n={n_patients}')
    print(f'{"="*60}')
    
    # Load split
    split_path = f'../outputs/tcga_brca/split_patient_{n_patients}.json'
    with open(split_path) as f:
        split = json.load(f)
    
    forget_idx = split['forget_indices']
    retain_idx = split['retain_indices']
    unseen_idx = split['unseen_indices']
    
    print(f'Forget: {len(forget_idx)}, Retain: {len(retain_idx)}, Unseen: {len(unseen_idx)}')
    
    # Compute matched negatives for this specific forget set
    matched_idx = compute_matched_negatives(forget_idx, unseen_idx, z_all, k=10)
    print(f'Matched negatives: {len(matched_idx)}')
    
    # Save matched negatives
    match_path = OUTPUT_DIR / f'matched_neg_n{n_patients}.json'
    with open(match_path, 'w') as f:
        json.dump({'matched_indices': matched_idx, 'k': 10, 'n_forget': len(forget_idx)}, f)
    
    # Baseline evaluation
    print('\n  Baseline:')
    baseline_res = evaluate_method(baseline_model, forget_idx, matched_idx, 'Baseline')
    
    # --- Method 1: Fisher scrubbing ---
    print('\n  Fisher scrubbing:')
    set_global_seed(GLOBAL_SEED)
    fisher_model = load_baseline()
    forget_ds = AnnDataDataset(adata, forget_idx)
    forget_loader = DataLoader(forget_ds, batch_size=128, shuffle=False)
    retain_ds = AnnDataDataset(adata, retain_idx)
    retain_loader = DataLoader(retain_ds, batch_size=128, shuffle=True)
    
    fisher = compute_fisher_diagonal(fisher_model, retain_loader, DEVICE, damping=0.1)
    fisher_scrub(fisher_model, forget_loader, fisher, DEVICE, scrub_lr=1e-4, scrub_steps=100)
    retain_finetune(fisher_model, retain_loader, DEVICE, epochs=10, lr=1e-4)
    fisher_res = evaluate_method(fisher_model, forget_idx, matched_idx, 'Fisher')
    
    # --- Method 2: Retain fine-tune ---
    print('\n  Retain fine-tune:')
    rt_dir = OUTPUT_DIR / f'retain_ft_n{n_patients}'
    rt_dir.mkdir(exist_ok=True)
    train_retain_finetune(
        baseline_checkpoint=BASELINE_CKPT_PATH, data_path=DATA_PATH,
        split_path=split_path, output_dir=str(rt_dir),
        epochs=50, lr=1e-4, batch_size=128, seed=GLOBAL_SEED)
    rt_model = load_baseline()
    rt_ckpt = torch.load(rt_dir / 'best_model.pt', map_location=DEVICE)
    rt_model.load_state_dict(rt_ckpt['model_state_dict'])
    rt_res = evaluate_method(rt_model, forget_idx, matched_idx, 'Retain-FT')
    
    # --- Method 3: Gradient ascent ---
    print('\n  Gradient ascent:')
    ga_dir = OUTPUT_DIR / f'ga_n{n_patients}'
    ga_dir.mkdir(exist_ok=True)
    train_gradient_ascent(
        baseline_checkpoint=BASELINE_CKPT_PATH, data_path=DATA_PATH,
        split_path=split_path, output_dir=str(ga_dir),
        ascent_steps=50, ascent_lr=1e-4,
        finetune_epochs=20, finetune_lr=1e-4,
        batch_size=128, seed=GLOBAL_SEED)
    ga_model = load_baseline()
    ga_ckpt = torch.load(ga_dir / 'best_model.pt', map_location=DEVICE)
    ga_model.load_state_dict(ga_ckpt['model_state_dict'])
    ga_res = evaluate_method(ga_model, forget_idx, matched_idx, 'Grad-ascent')
    
    all_patient_results[f'n={n_patients}'] = {
        'n_forget': len(forget_idx),
        'n_matched': len(matched_idx),
        'patient_ids': split.get('patient_ids', []),
        'baseline': baseline_res,
        'fisher_scrubbing': fisher_res,
        'retain_finetune': rt_res,
        'gradient_ascent': ga_res,
    }

print('\nAll patient-level experiments complete')


Patient-level forget set: n=5


Forget: 5, Retain: 920, Unseen: 164
Matched negatives: 30

  Baseline:


    Baseline: AUC=0.113, advantage=0.773

  Fisher scrubbing:
Computing Fisher diagonal on retain set...


  Computed Fisher diagonal over 920 retain samples
Fisher scrubbing for 100 steps...
  Step 1/100: forget_loss=1476.2666


  Step 20/100: forget_loss=122326233.6000


...

  Epoch  50: train=801.8849, val=634.4651 
Saved to ../outputs/tcga_brca/patient_unlearning/retain_ft_n20/best_model.pt


    Retain-FT: AUC=0.208, advantage=0.585

  Gradient ascent:
Data: torch.Size([1089, 978]), Forget: 20, Retain: 905, Device: cpu
Phase 1: Gradient ascent (50 steps, lr=0.0001)


  Start loss: 1451.41, End loss: 1493.25
Phase 2: Fine-tune (20 epochs max, lr=0.0001)


  Best val loss: 640.11, Epochs: 20
Saved to ../outputs/tcga_brca/patient_unlearning/ga_n20/best_model.pt


    Grad-ascent: AUC=0.216, advantage=0.568

All patient-level experiments complete


## Results Summary

In [5]:
# Save all results
results = {
    'dataset': 'TCGA-BRCA',
    'forget_type': 'patient_level',
    'forget_subtype': 'Basal',
    'results_by_size': all_patient_results,
}

with open(OUTPUT_DIR / 'patient_unlearning_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Print summary table
print('='*70)
print('TCGA-BRCA Patient-Level Unlearning Results')
print('='*70)
print(f'{"n":>3} {"Method":<20} {"AUC":>8} {"Advantage":>10}')
print('-'*70)

for size_key, size_res in all_patient_results.items():
    n = size_res['n_forget']
    for method in ['baseline', 'fisher_scrubbing', 'retain_finetune', 'gradient_ascent']:
        res = size_res[method]
        print(f'{n:3d} {method:<20} {res["auc"]:8.3f} {res["advantage"]:10.3f}')
    print('-'*70)

print(f'\nSaved to {OUTPUT_DIR / "patient_unlearning_results.json"}')

TCGA-BRCA Patient-Level Unlearning Results
  n Method                    AUC  Advantage
----------------------------------------------------------------------
  5 baseline                0.113      0.773
  5 fisher_scrubbing        0.187      0.627
  5 retain_finetune         0.160      0.680
  5 gradient_ascent         0.087      0.827
----------------------------------------------------------------------
 10 baseline                0.205      0.591
 10 fisher_scrubbing        0.166      0.668
 10 retain_finetune         0.207      0.586
 10 gradient_ascent         0.200      0.600
----------------------------------------------------------------------
 20 baseline                0.216      0.568
 20 fisher_scrubbing        0.211      0.578
 20 retain_finetune         0.208      0.585
 20 gradient_ascent         0.216      0.568
----------------------------------------------------------------------

Saved to ../outputs/tcga_brca/patient_unlearning/patient_unlearning_results.json


## Summary

| Output | Path |
|--------|------|
| All results | `outputs/tcga_brca/patient_unlearning/patient_unlearning_results.json` |
| Matched negatives | `outputs/tcga_brca/patient_unlearning/matched_neg_n{5,10,20}.json` |
| Method checkpoints | `outputs/tcga_brca/patient_unlearning/{method}_n{5,10,20}/best_model.pt` |

**Per experiment-runner skill:** Sequential numbering, seeds, JSON output, summary at end.

**Per mia-evaluation skill:** Per-split matched negatives (k=10), fresh attacker per evaluation, 69-dim v1, advantage metric.

**Next:** `35_tcga_brca_fisher_overlap.ipynb` - Fisher overlap analysis on TCGA-BRCA